<a href="https://colab.research.google.com/github/edmundzen/dengue-early-warning/blob/main/member4_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai

**MEMBER 4 : Gemini AI & Integration**

Integrate the Gemini API.
Generate AI recommendations/explanations based on the risk scores.
Help connect all the modules and test the complete workflow.
## Tasks Completed
- Connected to Google BigQuery.
- Read the `inspection_priority` table generated by Member 2.
- Integrated the Gemini API.
- Generated AI recommendations for high-risk dengue areas.
- Saved the recommendations as a CSV (`inspection_with_ai.csv`).
- Verified the output for dashboard integration.

## Workflow
BigQuery (inspection_priority)
⬇
Gemini API
⬇
AI Recommendations
⬇
CSV Output
⬇
Looker Studio Dashboard

## Step 1: Import Required Libraries

In [2]:
from google import genai
from google.genai import types
import pandas as pd
from google.cloud import bigquery
from google.colab import auth
from google.colab import auth, userdata

## Step 2: Authenticate with Google Cloud

In [3]:
auth.authenticate_user()

import subprocess
acct = subprocess.run(["gcloud","config","get-value","account"],
                      capture_output=True, text=True).stdout.strip()
print("Signed in as:", acct)

Signed in as: edmundzen83@gmail.com


## Step 3: Connect to BigQuery

In [4]:
PROJECT_ID = "dengue-early-warning"
client = bigquery.Client(project=PROJECT_ID)
print("Connected!")

Connected!


## Step 4: Load the `inspection_priority` Table

In [16]:
query = """
SELECT * FROM `dengue-early-warning.dengue_ew.inspection_priority`
WHERE risk_score > 0 ORDER BY risk_score DESC LIMIT 20
"""
inspection_df = client.query(query).to_dataframe()
print("Loaded", len(inspection_df), "cells")

Loaded 20 cells


In [17]:
print(inspection_df["risk_score"].tolist())

[24.73, 24.61, 22.84, 22.08, 21.92, 21.88, 21.85, 21.79, 21.77, 21.74, 21.54, 21.28, 21.27, 21.22, 21.14, 20.76, 20.63, 20.59, 20.57, 20.47]


## Step 5: Configure Gemini API

In [11]:
client_ai = genai.Client(api_key=userdata.get("GEMINI_KEY"))
print("Gemini ready")

Gemini ready


## Step 6: Generate AI Recommendations

In [19]:
inspection_df = inspection_df.head(3).copy()
inspection_df["AI_Recommendation"] = inspection_df.apply(generate_recommendation, axis=1)
print("Done:", len(inspection_df), "rows")
inspection_df[["cell_id","risk_score","AI_Recommendation"]]

  retry 1 (busy)...
  retry 2 (busy)...
  retry 3 (busy)...
  retry 1 (busy)...
  retry 2 (busy)...
  retry 3 (busy)...
  retry 1 (busy)...
  retry 2 (busy)...
  retry 3 (busy)...
Done: 3 rows


,cell_id,risk_score,AI_Recommendation
0,1.33875_103.86,24.73,[failed: 429 RESOURCE_EXHAUSTED. {'error': {'c...
1,1.33875_103.86,24.61,[failed: 429 RESOURCE_EXHAUSTED. {'error': {'c...
2,1.33875_103.85775,22.84,[failed: 429 RESOURCE_EXHAUSTED. {'error': {'c...


## Step 7: Export Recommendations to CSV

In [ ]:
inspection_df.to_csv("inspection_with_ai.csv", index=False)
print("CSV saved")

job = client.load_table_from_dataframe(
    inspection_df,
    "dengue-early-warning.dengue_ew.inspection_with_ai",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()
print("Uploaded", job.output_rows, "rows to inspection_with_ai")

# Conclusion

The Gemini AI module successfully converts dengue risk scores into actionable recommendations for health officials. These recommendations can be integrated into the dashboard to support faster and better decision-making during dengue outbreaks.